# Notebook 8 — Text Feature Engineering
Using `support_ticket_text` from `telecom_customers.csv` — free-text customer support
messages, a very common real-world unstructured field.

In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv("./telecom_customers.csv")
customers[["customer_id","support_ticket_text"]].head()

## Why Convert Text to Features?

ML models operate on numbers, not language. Text Feature Engineering is the bridge
between unstructured text and a model-consumable numeric representation — ranging
from simple hand-crafted statistics to full vector representations like TF-IDF.

## 1. Basic Text Statistics

In [ ]:
t = customers["support_ticket_text"]

customers["text_char_count"] = t.str.len()
customers["text_word_count"] = t.str.split().str.len()
customers["text_sentence_count"] = t.str.count(r"[.!?]+").clip(lower=1)
customers["text_digit_count"] = t.str.count(r"\d")
customers["text_special_char_count"] = t.str.count(r"[^a-zA-Z0-9\s]")
customers["text_uppercase_count"] = t.apply(lambda s: sum(1 for c in s if c.isupper()))
customers["text_lowercase_count"] = t.apply(lambda s: sum(1 for c in s if c.islower()))
customers["avg_word_length"] = customers["text_char_count"] / customers["text_word_count"]

customers[["support_ticket_text","text_char_count","text_word_count",
           "text_uppercase_count","text_special_char_count"]].head()

**Business meaning:** `text_uppercase_count` and `text_special_char_count` (like
repeated `!` or `?`) are cheap but effective **frustration/urgency signals** — a
ticket in all caps with multiple exclamation marks correlates strongly with dissatisfaction,
and by extension, churn risk.

## 2. Keyword Count Features (Domain Lexicon)

**When to use:** when you know specific words/phrases carry strong domain signal —
much cheaper and more interpretable than a full NLP pipeline for a targeted use case.

In [ ]:
churn_keywords = ["cancel", "expensive", "slow", "not working", "disconnect"]
satisfaction_keywords = ["thank", "great", "excellent", "resolved"]

def keyword_hits(text, keywords):
    text_l = text.lower()
    return sum(1 for kw in keywords if kw in text_l)

customers["churn_keyword_count"] = t.apply(lambda s: keyword_hits(s, churn_keywords))
customers["satisfaction_keyword_count"] = t.apply(lambda s: keyword_hits(s, satisfaction_keywords))

customers[["support_ticket_text","churn_keyword_count","satisfaction_keyword_count"]].head()

## 3. Bag of Words (BoW)

Represents text as raw word-frequency counts. **When to use:** simple baselines, or
when word *presence/frequency* alone is informative, without caring about how
distinctive a word is across the corpus.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=15, stop_words="english")
bow_matrix = bow.fit_transform(t)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=bow.get_feature_names_out())
bow_df.head()

## 4. TF-IDF (Term Frequency – Inverse Document Frequency)

**When to use:** almost always preferred over raw BoW for text classification —
TF-IDF down-weights common words that appear in most documents (less informative) and
up-weights rare, distinctive words (more informative).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=15, stop_words="english")
tfidf_matrix = tfidf.fit_transform(t)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())
tfidf_df.head()

## 5. N-Grams

Single words (unigrams) lose word-order context — "not working" vs "working" have
opposite meaning, but unigram BoW/TF-IDF treats "not" and "working" independently.
**Bigrams (2-word sequences)** capture short phrases like this directly.

In [ ]:
bigram_tfidf = TfidfVectorizer(max_features=10, ngram_range=(2,2), stop_words="english")
bigram_matrix = bigram_tfidf.fit_transform(t)
pd.DataFrame(bigram_matrix.toarray(), columns=bigram_tfidf.get_feature_names_out()).head()

## Summary — Feature Justification

| Feature | Source | Logic | Business Meaning | Leakage Risk | Decision |
|---|---|---|---|---|---|
| `churn_keyword_count` | support_ticket_text | lexicon match count | direct dissatisfaction signal | None | **Retain** |
| `text_uppercase_count` | support_ticket_text | char-level count | frustration/urgency proxy | None | **Retain** |
| TF-IDF top terms | support_ticket_text | TfidfVectorizer | distinctive vocabulary per ticket | None | **Retain** as a feature block, subject to Feature Selection |
| Raw BoW counts | support_ticket_text | CountVectorizer | word frequency | None | **Remove** in favor of TF-IDF (redundant, less informative) |

> **Note:** in a production pipeline, the `CountVectorizer`/`TfidfVectorizer` must be
> `.fit()` only on the training split, then `.transform()` on validation/test —
> otherwise vocabulary statistics leak information about the test set (Notebook 12/13).